# C05 Duplicate and cross-channel detection
Civic Operations · Nabeel Taresh · Group 23

## 1. Goal and contract

Group reports of the same incident. Output `NEW` / `DUPLICATE` / `RECURRENCE`.
Only OPEN jobs absorb a duplicate. After closure the same place is a recurrence.
Failure returns NEW. Signature: `run(facts, candidates) -> DuplicateResult`.

Read `05_Team_Workspace/C05_C06_changelog/RESULTS.md` for the report text.
The change log is `05_Team_Workspace/C05_C06_changelog/CHANGELOG.md`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "01_Application" / "data" / "sample").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "01_Application"))
import pandas as pd
pd.set_option("display.max_columns", 40)
print("repo:", ROOT)

## 2. Data

Real requests from `data/sample/requests_real_sample.csv` (or processed parquet).
Hand labels: about 300 pairs, **not in git yet**. Until they arrive, the harness
uses a **weak heuristic**. Those numbers are provisional. Do not quote them as
real results.

Labelling kit: `py 06_Workspace_Tools/c05_labelling.py sample` (seed 2026).

In [ ]:
sys.path.insert(0, str(ROOT / "06_Workspace_Tools"))
from c05_labelling import load_eval_pairs, load_requests
df, origin = load_requests("auto")
print("requests", len(df), "from", origin)
pairs, kind = load_eval_pairs()
print("eval pairs", len(pairs), "kind", kind)
print(pairs["label"].value_counts())
print(pairs["distance_bin"].value_counts() if "distance_bin" in pairs.columns else "no bins")

## 3. Baseline

`radius_time`: 150 m (ADDRESS) plus 48 h. `exact_key` needs BBL.

## 4. Proposed method

Production is `hybrid_evidence`: BBL key, radius/time, fuzzy address when pins
are weak, NetworkX group. Alternatives: DBSCAN, HDBSCAN, graph-only, fuzzy-only.
Ablation: `hybrid_no_fuzzy`.

## 5. Evaluation table

Same harness for every method: `py 06_Workspace_Tools/c05_c06_eval.py benchmark`.
If `label_kind` is `provisional_weak`, say so in the report.

In [ ]:
p = ROOT / "experiments" / "c05_benchmark.csv"
if p.exists():
    res = pd.read_csv(p)
    display(res)
else:
    print("Run: py 06_Workspace_Tools/c05_c06_eval.py benchmark")

## 6. Error analysis

False merge (predicted DUPLICATE, gold not DUPLICATE) is the costly error.
Slice by `address_type` and borough in the CSV. INTERSECTION pins are noisy.

## 7. Decision

Production: `hybrid_evidence` at VERSION 0.1.0. Pick by false-merge first, then
F1 and ARI. Revisit when hand labels exist.

## 8. Artifact and version

`components/c05_duplicate.py` · VERSION 0.1.0 ·
`match_evidence` reasons · changelog in `05_Team_Workspace/C05_C06_changelog/`.